In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [2]:
df_2011 = pd.read_csv("../data/clean/joins/2011_cleaned.csv")
df_2021 = pd.read_csv("../data/clean/joins/2021.csv")

In [3]:
df_2011.columns, len(df_2011.columns)

(Index(['area', 'mnemonics', 'all_usual_residents', 'white',
        'white_english_welsh_scottish_northern_irish_british', 'white_irish',
        'white_gypsy_or_irish_traveller', 'white_other_white',
        'mixed_multiple_ethnic_groups',
        'mixed_multiple_ethnic_groups_white_and_black_caribbean',
        'mixed_multiple_ethnic_groups_white_and_black_african',
        'mixed_multiple_ethnic_groups_white_and_asian',
        'mixed_multiple_ethnic_groups_other_mixed', 'asian_asian_british',
        'asian_asian_british_indian', 'asian_asian_british_pakistani',
        'asian_asian_british_bangladeshi', 'asian_asian_british_chinese',
        'asian_asian_british_other_asian',
        'black_african_caribbean_black_british',
        'black_african_caribbean_black_british_african',
        'black_african_caribbean_black_british_caribbean',
        'black_african_caribbean_black_british_other_black',
        'other_ethnic_group', 'other_ethnic_group_arab',
        'other_ethnic_grou

In [4]:
df_2021.columns, len(df_2021.columns)

(Index(['area', 'mnemonics', 'total_all_usual_residents',
        'asian,_asian_british_or_asian_welsh',
        'asian,_asian_british_or_asian_welsh_bangladeshi',
        'asian,_asian_british_or_asian_welsh_chinese',
        'asian,_asian_british_or_asian_welsh_indian',
        'asian,_asian_british_or_asian_welsh_pakistani',
        'asian,_asian_british_or_asian_welsh_other_asian',
        'black,_black_british,_black_welsh,_caribbean_or_african',
        'black,_black_british,_black_welsh,_caribbean_or_african_african',
        'black,_black_british,_black_welsh,_caribbean_or_african_caribbean',
        'black,_black_british,_black_welsh,_caribbean_or_african_other_black',
        'mixed_or_multiple_ethnic_groups',
        'mixed_or_multiple_ethnic_groups_white_and_asian',
        'mixed_or_multiple_ethnic_groups_white_and_black_african',
        'mixed_or_multiple_ethnic_groups_white_and_black_caribbean',
        'mixed_or_multiple_ethnic_groups_other_mixed_or_multiple_ethnic_gro

In [5]:
# rename maps to harmonise 2011 and 2021 columns
rename_2011 = {
    'all_usual_residents': 'total_pop',
    'all_categories_occupation': 'total_employed',
    'white': 'eth_white',
    'asian_asian_british': 'eth_asian',
    'black_african_caribbean_black_british': 'eth_black',
    'mixed_multiple_ethnic_groups': 'eth_mixed',
    'other_ethnic_group': 'eth_other',
    'very_good_health': 'health_very_good',
    'good_health': 'health_good',
    'fair_health': 'health_fair',
    'bad_health': 'health_bad',
    'very_bad_health': 'health_very_bad',
    '1._managers,_directors_and_senior_officials': 'occ_managers',
    '2._professional_occupations': 'occ_professionals',
    '3._associate_professional_and_technical_occupations': 'occ_associate_prof',
    '4._administrative_and_secretarial_occupations': 'occ_admin',
    '5._skilled_trades_occupations': 'occ_skilled_trades',
    '6._caring,_leisure_and_other_service_occupations': 'occ_caring',
    '7._sales_and_customer_service_occupations': 'occ_sales',
    '8._process_plant_and_machine_operatives': 'occ_machine_ops',
    '9._elementary_occupations': 'occ_elementary',
}

rename_2021 = {
    'total_all_usual_residents': 'total_pop',
    'total_all_usual_residents_aged_16_years_and_over_in_employment_the_week_before_the_census_x': 'total_employed',
    'white': 'eth_white',
    'asian,_asian_british_or_asian_welsh': 'eth_asian',
    'black,_black_british,_black_welsh,_caribbean_or_african': 'eth_black',
    'mixed_or_multiple_ethnic_groups': 'eth_mixed',
    'other_ethnic_group': 'eth_other',
    'very_good_health': 'health_very_good',
    'good_health': 'health_good',
    'fair_health': 'health_fair',
    'bad_health': 'health_bad',
    'very_bad_health': 'health_very_bad',
    '1._managers,_directors_and_senior_officials': 'occ_managers',
    '2._professional_occupations': 'occ_professionals',
    '3._associate_professional_and_technical_occupations': 'occ_associate_prof',
    '4._administrative_and_secretarial_occupations': 'occ_admin',
    '5._skilled_trades_occupations': 'occ_skilled_trades',
    '6._caring,_leisure_and_other_service_occupations': 'occ_caring',
    '7._sales_and_customer_service_occupations': 'occ_sales',
    '8._process,_plant_and_machine_operatives': 'occ_machine_ops',
    '9._elementary_occupations': 'occ_elementary',
}


df_2011 = df_2011.rename(columns=rename_2011)
df_2021 = df_2021.rename(columns=rename_2021)


core_cols = list(rename_2011.values())
df_2011_clean = df_2011[['area', 'mnemonics'] + core_cols].copy()
df_2021_clean = df_2021[['area', 'mnemonics'] + core_cols].copy()

df_2011_clean['Year'] = 2011
df_2021_clean['Year'] = 2021

df_master = pd.concat([df_2011_clean, df_2021_clean], ignore_index=True)


# build proportion columns without overwriting the raw counts
pop_cols = ['eth_white', 'eth_asian', 'eth_black', 'eth_mixed', 'eth_other',
            'health_very_good', 'health_good', 'health_fair', 'health_bad', 'health_very_bad']

emp_cols = ['occ_managers', 'occ_professionals', 'occ_associate_prof', 'occ_admin',
            'occ_skilled_trades', 'occ_caring', 'occ_sales', 'occ_machine_ops', 'occ_elementary']

for col in pop_cols:
    df_master[col + '_pct'] = df_master[col] / df_master['total_pop']

for col in emp_cols:
    df_master[col + '_pct'] = df_master[col] / df_master['total_employed']


features = [c + '_pct' for c in pop_cols + emp_cols]


# missingness and sanity check
print("missing values per feature:")
print(df_master[features].isna().sum())
print("rows with any nan:", df_master[features].isna().any(axis=1).sum())
print("any negative proportions:", (df_master[features] < 0).any().any())
print("any proportions above 1:", (df_master[features] > 1).any().any())


# drop any rows with nans before fitting
df_model = df_master.dropna(subset=features).copy()


# standardise
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_model[features])


# pca
pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)
df_model['PCA_1'] = pca_components[:, 0]
df_model['PCA_2'] = pca_components[:, 1]

print("\npca explained variance ratio:", pca.explained_variance_ratio_)
print("pc1 + pc2 combined:", pca.explained_variance_ratio_.sum())

loadings = pd.DataFrame(pca.components_.T, index=features, columns=['PC1', 'PC2'])
print("\nloadings sorted by pc1:")
print(loadings.sort_values('PC1'))


# tsne
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
tsne_components = tsne.fit_transform(X_scaled)
df_model['TSNE_1'] = tsne_components[:, 0]
df_model['TSNE_2'] = tsne_components[:, 1]


# save outputs
df_model.to_csv('PCA.csv', index=False)
loadings.to_csv('PCA_loadings.csv')

missing values per feature:
eth_white_pct             0
eth_asian_pct             0
eth_black_pct             0
eth_mixed_pct             0
eth_other_pct             0
health_very_good_pct      0
health_good_pct           0
health_fair_pct           0
health_bad_pct            0
health_very_bad_pct       0
occ_managers_pct          0
occ_professionals_pct     0
occ_associate_prof_pct    0
occ_admin_pct             0
occ_skilled_trades_pct    0
occ_caring_pct            0
occ_sales_pct             0
occ_machine_ops_pct       0
occ_elementary_pct        0
dtype: int64
rows with any nan: 0
any negative proportions: False
any proportions above 1: False

pca explained variance ratio: [0.49101816 0.19910209]
pc1 + pc2 combined: 0.6901202443299552

loadings sorted by pc1:
                             PC1       PC2
health_fair_pct        -0.304328 -0.019021
occ_caring_pct         -0.262000 -0.062425
occ_machine_ops_pct    -0.256532 -0.161794
health_bad_pct         -0.253646 -0.174644
occ_skill